# Telecom Customer Churn Prediction
This notebook covers loading, cleaning, exploring, training, and evaluating machine learning models to predict customer churn using the Telco Churn dataset.

## Step 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc, roc_auc_score

In [ ]:
# Load dataset
dataset = pd.read_csv("Telco-Customer-Churn.csv")
dataset.head()

In [ ]:
dataset.tail()

In [ ]:
dataset.info()

In [ ]:
dataset.isna().sum()

In [ ]:
dataset.describe()

## Step 2: Clean Data & Feature Engineering

In [ ]:
# Clean TotalCharges (coerce empty spaces to NaN and fill with median)
dataset['TotalCharges'] = pd.to_numeric(dataset['TotalCharges'], errors='coerce')
dataset['TotalCharges'] = dataset['TotalCharges'].fillna(dataset['TotalCharges'].median())

# Drop customerID column
dataset.drop(columns=['customerID'], inplace=True, errors='ignore')

# Engineer AvgMonthlyCharge
dataset['AvgMonthlyCharge'] = dataset.apply(
    lambda row: row['TotalCharges'] / row['tenure'] if row['tenure'] > 0 else row['MonthlyCharges'], 
    axis=1
)
dataset[['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge']].head()

## Step 3: Encode Categorical Columns

In [ ]:
# Encode object types to numbers using LabelEncoder
label_encoders = {}
for col in dataset.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    dataset[col] = le.fit_transform(dataset[col])
    label_encoders[col] = le
dataset.head()

## Step 4: Split Features and Target

In [ ]:
X = dataset.drop('Churn', axis=1)
y = dataset['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Step 5: Scale Numeric Features

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
X_train_scaled.head()

## Step 6: Train and Compare Models

In [ ]:
# Model 1: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

# Model 3: HistGradientBoosting Classifier
gb_model = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_proba = gb_model.predict_proba(X_test_scaled)[:, 1]

## Step 7: Evaluate the Models

In [ ]:
print("--- Accuracy Comparison ---")
print(f"Random Forest:       {accuracy_score(y_test, rf_pred):.4f}")
print(f"Logistic Regression: {accuracy_score(y_test, lr_pred):.4f}")
print(f"Gradient Boosting:   {accuracy_score(y_test, gb_pred):.4f}")

print("\n--- ROC-AUC Comparison ---")
print(f"Random Forest:       {roc_auc_score(y_test, rf_proba):.4f}")
print(f"Logistic Regression: {roc_auc_score(y_test, lr_proba):.4f}")
print(f"Gradient Boosting:   {roc_auc_score(y_test, gb_proba):.4f}")

In [ ]:
# Classification Report and Confusion Matrix for Best Model (Logistic Regression)
print("Classification Report (Logistic Regression):")
print(classification_report(y_test, lr_pred))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion Matrix - Logistic Regression")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Actual")

# ROC Curve
for model_name, proba in [('Random Forest', rf_proba), ('Logistic Regression', lr_proba), ('Gradient Boosting', gb_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax[1].plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

ax[1].plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curves')
ax[1].legend()
plt.show()

In [ ]:
# Plot Feature Importance for Random Forest as an extra insight
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances (Random Forest)")
sns.barplot(x=importances[indices], y=features[indices], palette="viridis")
plt.tight_layout()
plt.show()